# 🎬 OpenShorts + PeakCut UI: AI Shorts Generator (GPU Accelerated)
Use this repository on Google Colab with **free NVIDIA GPU** acceleration to turn long videos into viral 9:16 shorts.

### ⚡ Highlights:
- **PeakCut Studio UI**: Full dark-warm web dashboard accessible via a free public Cloudflare Tunnel link.
- **Smart Processing**: Active speaker face tracking, 2-person split layout, dead silence jump-cutting (Smart Cut), and bouncy subtitles.
- **Multi-Source**: Supports YouTube, Twitch, Kick live streams & VODs, and direct video uploads.

In [ ]:
# @title 1. Verify NVIDIA GPU Acceleration
!nvidia-smi

In [ ]:
# @title 2. Setup Current Repository
# If running from Google Drive or an uploaded archive of this repo:
import os
if os.path.exists('/content/drive/MyDrive/yt'):
    %cd /content/drive/MyDrive/yt
elif os.path.exists('/content/yt'):
    %cd /content/yt

print('Working directory:', os.getcwd())
print('Files available:', os.listdir('.'))

In [ ]:
# @title 3. Install System Dependencies & Cloudflared Tunnel
!apt-get update -qq
!apt-get install -y -qq ffmpeg libsm6 libxext6 libgl1 fonts-freefont-ttf

# Download Cloudflared for instant zero-config HTTPS public tunnel
!wget -q -nc -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print('✅ System dependencies and cloudflared ready!')

In [ ]:
# @title 4. Install Python Requirements
!pip install -q -r requirements.txt
!pip install -q uvicorn fastapi requests
print('✅ Python dependencies installed!')

In [ ]:
# @title 5. API Keys (Optional)
import os
# Paste your Gemini API key below, or leave empty for self-hosted zero-key mode
GEMINI_API_KEY = "" # @param {type:"string"}
if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    print('✅ GEMINI_API_KEY configured!')
else:
    print('ℹ️ Running in self-hosted zero-key mode.')

In [ ]:
# @title 6. Launch PeakCut UI & Start Making Shorts
import subprocess
import time
import re

# Start Cloudflared tunnel pointing to port 8000
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Extract public HTTPS URL
public_url = None
for _ in range(40):
    line = tunnel.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

print('=' * 65)
if public_url:
    print(f'🎉 PEAKCUT STUDIO IS LIVE AT: {public_url}')
    print('👉 Click the link above to open PeakCut Studio in your browser!')
else:
    print('⚠️ Tunnel starting... check logs below.')
print('=' * 65)

# Start OpenShorts FastAPI server
!python -m uvicorn app:app --host 127.0.0.1 --port 8000